In [ ]:
import os
from pathlib import Path
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean
import contextily as ctx
import glob
import scienceplots

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ──────────────────────────────────────────────────────────────────────────────

POSTPROC_DIR = Path("/export/lv9/projects/dws/results/graphs/benthos/")

OBS_CSV = {
    "SIBES": POSTPROC_DIR / "feeding_groups_gC_m2_ecotopes_SIBES.csv",
    "SUBES": POSTPROC_DIR / "feeding_groups_gC_m2_ecotopes_SUBES.csv",
}

FG_OBS_COLS   = ["epibenthos_Y1_c","deposit_feeders_Y2_c",
             "suspension_feeders_Y3_c","endobenthos_Y5_c"]
FG_LABELS = ["Epibenthos (Y1)","Deposit feeders (Y2)",
             "Suspension feeders (Y3)","Endobenthos (Y5)"]

fg, label = FG_COLS[3], FG_LABELS[3]

# Fixed map window: Dutch Wadden Sea to Ems-Dollard
MAP_EXTENT_LONLAT = (4.65, 6.65, 52.85, 53.6)

# Fixed color scale configuration
VMIN = 1e-3
VMAX = 1e+1
log_norm = mcolors.LogNorm(vmin=VMIN, vmax=VMAX)

sibes = pd.read_csv(OBS_CSV["SIBES"])
subes = pd.read_csv(OBS_CSV["SUBES"])

sibes_mean = (sibes.groupby(["sampling_station_id"])[["x","y"] + FG_COLS]
             .mean().reset_index())
subes_mean = (subes.groupby(["sampling_station_id"])[["x","y"] + FG_COLS]
             .mean().reset_index())

# --- Plotting ---
fig = plt.figure(figsize=(9, 6))
ax = plt.axes(projection=ccrs.Mercator())
ax.set_extent(MAP_EXTENT_LONLAT, crs=ccrs.PlateCarree())

## Add map features
#ax.add_feature(cfeature.LAND, zorder=2, edgecolor='k', facecolor='lightgray')
#ax.add_feature(cfeature.COASTLINE, zorder=2)

# Add the high-resolution background
ctx.add_basemap(ax, 
                crs=ax.projection.to_string(), 
                #source=ctx.providers.CartoDB.Positron,
                source=ctx.providers.Esri.WorldShadedRelief,
                zorder=1,
               )

txt = ax.texts[-1]
txt.set_position([0.99,0.02])
txt.set_ha('right')

s1 = ax.scatter(sibes_mean["x"].values, sibes_mean["y"].values, c = sibes_mean[fg].values,
                transform=ccrs.PlateCarree(),
                cmap=cmocean.cm.matter,
                norm=log_norm,
                s=2.5, marker="s", linewidths=0.0,
                zorder=2
                )

s2 = ax.scatter(subes_mean["x"].values, subes_mean["y"].values, c = subes_mean[fg].values,
                transform=ccrs.PlateCarree(),
                cmap=cmocean.cm.matter,
                norm=log_norm,
                s=8, marker="s", linewidths=0.0,
                zorder=3
                )

# Formatting
cbar = plt.colorbar(s1, ax=ax, shrink=0.75, pad=0.05)
cbar.ax.tick_params(labelsize=10)
cbar.set_label(r"Biomass (gC m$^{-2}$)", fontsize=12)
plt.title(f"{label} | SIBES/SUBES station means", fontsize=12, pad=10)

fig_path = os.path.join(POSTPROC_DIR,f"{fg}_all_mean.png")
plt.savefig(fig_path, dpi=600, bbox_inches='tight')
plt.close(fig)


